# ThemeFinder with SystemOne: minimal example

This notebook runs the hybrid pipeline (`find_themes_hybrid`) on the bundled example data and shows three things:

1. the **input** data structure,
2. the **pipeline** structure and what the SystemOne stages receive,
3. the **output** structure, including the probabilities.

Requirements: `pip install 'themefinder[systemone]'`, an OpenAI key (`OPENAI_API_KEY`) for the generative stages, and a TypeSafe key (`TYPESAFE_API_KEY`) for the classification stages.

In [ ]:
import pandas as pd

from themefinder import OpenAILLM, SystemOne, find_themes_hybrid

## 1. Input structure

The pipeline takes a DataFrame with two columns, `response_id` and `response`, plus the survey question the responses answer.

In [ ]:
question = "What improvements would you like to see in local bus services?"

responses = pd.read_json("example_data.json")
responses.head()

## 2. Pipeline structure

Five stages. The first three write text and run on the LLM; the last two are decisions and run on SystemOne, merged into a single request per batch of responses:

| Stage | What it does | Backend |
|---|---|---|
| 1. Theme generation | draft themes from the responses | LLM |
| 2. Theme condensation | merge duplicate themes | LLM |
| 3. Theme refinement | finalise the theme list | LLM |
| 4. Theme mapping | label each response with themes | SystemOne |
| 5. Detail detection | flag evidence-rich responses | SystemOne |

The SystemOne stages send one request per 20 responses. Shared context is stated once in the request `state`; every question is a small JSON pointer into it, and each answer is a single probability:

```
state:     question | topics {topic_id: text} | judgement rubrics | responses [x20]
questions: r<id>_theme_<topic> x n_themes | r<id>_gives_reason | r<id>_evidence_rich
answers:   {"r1_theme_A": {"noul": 0.987}, ...}   # one probability per question
```

Code then thresholds the probabilities: themes at or above 0.5 become `labels` (with "Other" / "No Reason Given" fallbacks), and `evidence_rich` is "YES" at or above the evidence threshold (default 0.16).

In [ ]:
# The generative stages use any OpenAI model; the classification stages use jev.
llm = OpenAILLM(model="gpt-4o-mini", request_kwargs={"temperature": 0})
systemone_client = SystemOne.from_env()  # reads TYPESAFE_API_KEY

In [ ]:
results = await find_themes_hybrid(responses, llm, systemone_client, question)

## 3. Output structure

Identical shape to `find_themes`, with probability columns added by the SystemOne stages.

In [ ]:
# The refined themes: topic_id, topic (label: description), source_topic_count
results["themes"]

In [ ]:
# Theme mapping: labels per response, plus the full probability per theme.
# Probabilities are kept, so thresholds can be re-applied later without re-querying.
results["mapping"].head()

In [ ]:
# Detail detection: evidence flag per response, plus its probability
results["detailed_responses"].head()

## Classification stages on their own

`classify_responses_systemone` runs just the SystemOne part against an existing theme list: a DataFrame with `topic_id` and `topic` columns. Useful when themes are already agreed and only the classification should be re-run.

In [ ]:
from themefinder import classify_responses_systemone

themes = pd.DataFrame(
    {
        "topic_id": ["A", "B"],
        "topic": [
            "Frequency: buses should run more often",
            "Reliability: buses should arrive when scheduled",
        ],
    }
)

classified, unprocessable = await classify_responses_systemone(
    responses, systemone_client, question=question, refined_themes_df=themes
)
classified.head()